# Loan Underwriting - Multi-Agent Demo
Manager-worker delegation on Semantic Kernel + Groq.

In [7]:
!pip install -q semantic-kernel openai
import os, sys
sys.path.insert(0, '.')
# os.environ['GROQ_API_KEY'] = 'your_key_here' # optional; leave unset to run offline

In [8]:
from run_demo import CASES
from agents.manager import UnderwritingManager
from state import print_trace
manager = UnderwritingManager()

In [9]:
rec = await manager.underwrite(CASES['sanction'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 124}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Ravi Kumar. Net monthly income Rs 60,000. Existing EMIs Rs 5,000. Requesting a loan of Rs 5,00,000 for 36 months."}, "re
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 60000.0}, "result": true, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "EmiAgent"}
[EmiAgent            ] tool:compute_emi             {"args": {"principal": 500000.0, "annual_rate": 10.5, "months": 36}, "result": 16251.22, "via": "kernel"}
[EmiAgent            ] tool:total_interest          {"args": {"emi": 16251.22, "months": 36, "principal": 500000.0}, "result": 85043.92, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "AffordabilityAgent"}
[AffordabilityAgent  ] tool:emi_to_income_ratio     {"args": {"emi": 16251.

In [10]:
rec = await manager.underwrite(CASES['decline (unaffordable)'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 125}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Neha Singh. Net monthly income Rs 30,000. Existing EMIs Rs 5,000. Requesting a loan of Rs 10,00,000 for 24 months."}, "r
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 30000.0}, "result": true, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "EmiAgent"}
[EmiAgent            ] tool:compute_emi             {"args": {"principal": 1000000.0, "annual_rate": 10.5, "months": 24}, "result": 46376.04, "via": "kernel"}
[EmiAgent            ] tool:total_interest          {"args": {"emi": 46376.04, "months": 24, "principal": 1000000.0}, "result": 113024.96, "via": "kernel"}
[UnderwritingManager ] delegate                     {"to": "AffordabilityAgent"}
[AffordabilityAgent  ] tool:emi_to_income_ratio     {"args": {"emi": 463

In [11]:
rec = await manager.underwrite(CASES['refer (income missing)'])
print_trace(rec)

[UnderwritingManager ] received_application         {"chars": 94}
[UnderwritingManager ] delegate                     {"to": "DocumentAgent"}
[DocumentAgent       ] tool:extract_fields          {"args": {"text": "Applicant: Meera Shah. Existing EMIs Rs 3,000. Requesting a loan of Rs 3,00,000 for 24 months."}, "result": {"name": "Meera Shah", 
[DocumentAgent       ] tool:validate_income_present {"args": {"income": 0.0}, "result": false, "via": "kernel"}
[UnderwritingManager ] halt                         {"missing": ["income"]}
[UnderwritingManager ] tool:decide_rule             {"args": {"income_ok": false, "affordable": false, "defaults": 0, "missed": 0}, "result": {"decision": "refer", "reasons": ["required data missing (e.
[UnderwritingManager ] tool:format_justification    {"args": {"decision": "refer", "reasons": "required data missing (e.g. income) - not assumed, referred to human underwriter", "emi": 0.0, "ratio": 0.0
[UnderwritingManager ] final_decision               {"decisio

In [12]:
print(''.join(open('run_log.jsonl').readlines()[-8:]))

{"ts": 1795090508.489, "agent": "UnderwritingManager", "event": "received_application", "data": {"chars": 94}}
{"ts": 1795090508.49, "agent": "UnderwritingManager", "event": "delegate", "data": {"to": "DocumentAgent"}}
{"ts": 1795090508.49, "agent": "DocumentAgent", "event": "tool:extract_fields", "data": {"args": {"text": "Applicant: Meera Shah. Existing EMIs Rs 3,000. Requesting a loan of Rs 3,00,000 for 24 months."}, "result": {"name": "Meera Shah", "income": null, "existing_emis": 3000.0, "amount": 300000.0, "tenure_months": 24, "annual_rate": null}, "via": "kernel"}}
{"ts": 1795090508.49, "agent": "DocumentAgent", "event": "tool:validate_income_present", "data": {"args": {"income": 0.0}, "result": false, "via": "kernel"}}
{"ts": 1795090508.491, "agent": "UnderwritingManager", "event": "halt", "data": {"missing": ["income"]}}
{"ts": 1795090508.491, "agent": "UnderwritingManager", "event": "tool:decide_rule", "data": {"args": {"income_ok": false, "affordable": false, "defaults": 0, 